# 06. Dual-Engine Anomaly Detection & Predictive Fault Risk Modeling

## Electrical Engineering Context
Detecting early thermal degradation, partial discharge, or abnormal harmonic loading before an electrical trip occurs is a core objective of modern distribution asset management.
In this notebook, we implement and benchmark:
1. **Method 1 — Statistical Anomaly Engine:**
   Z-scores $(|Z| > 3.0)$ and IQR outlier bounds with explicit parameter attribution (e.g. Current Spike, Voltage Sag, Thermal Hotspot).
2. **Method 2 — Machine Learning Anomaly Engine:**
   Multivariate **Isolation Forest** operating across voltage, current, PF, imbalance, loss %, and equipment temperature.
3. **Consensus Analysis:**
   Comparing statistical vs ML agreement.
4. **Predictive Fault Risk Modeling (Section 13):**
   Evaluating whether operational indicators can forecast faults within a 2-hour forward horizon using temporal train-test splitting.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.utils.config import PROCESSED_DATA_DIR
from src.anomaly_detection.detector import FaultPredictiveModel

df_meas = pd.read_csv(PROCESSED_DATA_DIR / "fact_electrical_measurements.csv")
print(f"Loaded {len(df_meas):,} telemetry records.")


## 1. Anomaly Detection Engine Consensus Breakdown
We compare how many records were flagged by Statistical Z-Score only, Isolation Forest only, or Dual-Method confirmed.


In [ ]:
consensus_summary = df_meas["anomaly_consensus"].value_counts()
print("Anomaly Consensus Summary:")
print(consensus_summary)

plt.figure(figsize=(9, 4.5))
df_meas["anomaly_consensus"].value_counts().plot(kind="barh", color=["#2d3748", "#dd6b20", "#3182ce", "#e53e3e"])
plt.title("Statistical vs Machine Learning Anomaly Consensus Tiers", fontsize=12, fontweight="bold")
plt.xlabel("Record Count")
plt.tight_layout()
plt.show()


## 2. Statistical Anomaly Causal Attribution
Why were statistical outliers flagged? We examine the engineering root-causes.


In [ ]:
stat_reasons = df_meas[df_meas["stat_anomaly_flag"] == 1]["stat_anomaly_reason"].value_counts().head(10)
print("Top 10 Statistical Anomaly Drivers:")
print(stat_reasons)


## 3. Predictive Fault Risk Modeling (Temporal Train-Test Split)
We evaluate forward fault prediction over the next 2-hour window using Logistic Regression and Random Forest.


In [ ]:
predictor = FaultPredictiveModel(prediction_horizon_intervals=4)
results = predictor.train_and_evaluate(df_meas)

print(f"Total Test Samples: {results['total_test_samples']:,}")
print(f"Actual Faults in Test Horizon: {results['actual_faults_in_test']}")
print("\n--- RANDOM FOREST CLASSIFICATION PERFORMANCE ---")
print(f"Precision : {results['random_forest']['precision']:.4f}")
print(f"Recall    : {results['random_forest']['recall']:.4f}")
print(f"F1-Score  : {results['random_forest']['f1_score']:.4f}")
print(f"ROC-AUC   : {results['random_forest']['roc_auc']:.4f}")
print(f"Confusion Matrix: {results['random_forest']['confusion_matrix']}")

print("\nLeading Operational Predictors of Imminent Fault Risk:")
for feat, imp in list(results['random_forest']['feature_importances'].items())[:5]:
    print(f"  * {feat:<28}: {imp:.4f}")
